# Targeted rebalance — surgical fill of underfilled cells

Reads existing confirmed_seeds.json, identifies cells under target_per_cell, samples candidates ONLY from those cells via direct FairFace metadata lookup (no random stratified resample), runs prefilter on just those, appends to balanced list.


In [ ]:
%pip install --force-reinstall --no-deps mediapipe==0.10.20
%pip install ftfy regex torchsde
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os, sys
from pathlib import Path
os.environ['HF_HOME'] = '/local_disk0/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/local_disk0/hf_cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TORCH_EXTENSIONS_DIR'] = '/dev/shm/torch_extensions'
os.environ['TMPDIR'] = '/dev/shm/tmp'
Path('/dev/shm/torch_extensions').mkdir(parents=True, exist_ok=True)
Path('/dev/shm/tmp').mkdir(parents=True, exist_ok=True)
HF_TOKEN = dbutils.secrets.get(scope='cap-secrets', key='hf-token')
os.environ['HF_TOKEN'] = HF_TOKEN
PULID_SRC = '/Volumes/ds_work/alenj00/cap_cache/pulid_src'
if PULID_SRC not in sys.path: sys.path.insert(0, PULID_SRC)

In [ ]:
from click.testing import CliRunner
from cap.cli.targeted_rebalance import main as tr_cli
result = CliRunner().invoke(tr_cli, [
  '--config', '/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline/configs/full.yaml',
  '--confirmed-seeds-file', '/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter/confirmed_seeds.json',
  '--output-dir', '/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter',
  '--target-per-cell', '14',
  '--max-attempts-per-cell', '30',
  '--cache-dir', '/local_disk0/hf_cache',
  '--pulid-src', PULID_SRC,
  '--hf-token', HF_TOKEN,
], catch_exceptions=False, standalone_mode=False)
print(result.output[-3000:])

In [ ]:
import json
from pathlib import Path
out = Path('/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter/confirmed_seeds_balanced.json')
ids = json.loads(out.read_text())
msg = f'Targeted rebalance complete: {len(ids)} balanced confirmed seeds'
print(msg)
dbutils.notebook.exit(msg)